# Indy counts preprocessing

## Dataset and output contract

The source is the Indy subset of [Zenodo record 3854034](https://zenodo.org/records/3854034): 37 recording sessions containing per-electrode spike-event timestamps and 250 Hz fingertip kinematics. Raw MAT files are immutable and are only read by this notebook.

The model-ready format is intentionally minimal:

- **Model input:** unsmoothed spike-event counts from the 96 M1 electrodes, one count per electrode per 40 ms bin. Counts are stored as `uint8`; the downstream model may select 32 of the 96 electrodes and compute causal features such as EWMA.
- **Training target:** two-dimensional fingertip velocity `(-x, -y)` in cm/s, stored as `float32`.
- **Provenance:** session name, source MD5, bin width, schema version, and causal target-method scalars.
- Cursor position, target position, fingertip position, waveform snippets, channel-area strings, unit counts, absolute timestamps, plots, and exploratory results are not copied into the model-ready artifacts.

Processing is strictly causal. Spike events are counted inside completed 40 ms bins. At each bin end, the latest already-observed kinematic sample is selected, followed by a forward-only 3 Hz Butterworth filter and backward-difference velocity. No centered interpolation, zero-phase filtering, central difference, smoothing of future neural samples, normalization, or channel selection is applied here.

## Fixed chronological split

| Split | Sessions | Dates | Share |
|---|---:|---|---:|
| Train | 29 | 2016-04 through 2016-10 | 78.4% |
| Validation | 4 | 2016-12 | 10.8% |
| Test | 4 | 2017-01 | 10.8% |

The split is session-level; time bins are never randomly mixed across splits. January 2017 is the locked test month.

```text
data/processed/indy_loco/indy/
  train/        29 session NPZ files
  validation/    4 session NPZ files
  test/          4 session NPZ files
  session_inventory.csv
  manifest.json
```

This notebook is delivered without executed outputs. Run it from top to bottom with the project virtual environment selected as the kernel.

In [12]:
from __future__ import annotations

import hashlib
import json
import sys
import time
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import yaml
from IPython.display import display


def find_repo_root() -> Path:
    'Locate the repository from either the root or this notebook directory.'
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs/datasets/indy_sessions.yaml").exists() and (
            candidate / "src/intent_decoder"
        ).exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside Summer-Hand-intent-decoder-proj")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.intent_decoder.features.causal import causal_sample_hold, causal_velocity

RAW_DIR = ROOT / "data/raw/indy_loco/indy"
OUTPUT_DIR = ROOT / "data/processed/indy_loco/indy"
CONFIG_PATH = ROOT / "configs/datasets/indy_sessions.yaml"

BIN_S = 0.040
VELOCITY_LOWPASS_HZ = 3.0
SCHEMA_VERSION = "indy_counts_velocity_v2"
VERIFY_MD5 = True
OVERWRITE = True

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    registry = yaml.safe_load(handle)

EXPECTED_MD5 = dict(registry["official_indy_sessions"])
EXPECTED_SESSIONS = list(EXPECTED_MD5)
SPLITS = {name: list(sessions) for name, sessions in registry["chronological_split"].items()}
SPLIT_FOR_SESSION = {
    session: split for split, sessions in SPLITS.items() for session in sessions
}
EXPECTED_CHANNEL_NAMES = np.asarray([f"M1 {index:03d}" for index in range(1, 97)])

assert len(EXPECTED_SESSIONS) == 37
assert {name: len(sessions) for name, sessions in SPLITS.items()} == {
    "train": 29,
    "validation": 4,
    "test": 4,
}
flattened_split = [session for sessions in SPLITS.values() for session in sessions]
assert len(flattened_split) == len(set(flattened_split)) == 37
assert set(flattened_split) == set(EXPECTED_SESSIONS)

print(f"Raw input: {RAW_DIR}")
print(f"Processed output: {OUTPUT_DIR}")
print("Split counts:", {name: len(sessions) for name, sessions in SPLITS.items()})

Raw input: /Users/yinzhecheng/Documents/UT/SECOND YEAR/STM32_Research/Summer-Hand-intent-decoder-proj/data/raw/indy_loco/indy
Processed output: /Users/yinzhecheng/Documents/UT/SECOND YEAR/STM32_Research/Summer-Hand-intent-decoder-proj/data/processed/indy_loco/indy
Split counts: {'train': 29, 'validation': 4, 'test': 4}


## 1. Verify the raw collection

All 37 official filenames must be present. When `VERIFY_MD5=True`, every MAT file is checked against the Zenodo checksum before processing starts.

In [13]:
def digest_file(path: Path, algorithm: str = "md5", chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.new(algorithm)
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


raw_rows = []
actual_names = {path.stem for path in RAW_DIR.glob("indy_*.mat")}

for index, session in enumerate(EXPECTED_SESSIONS, start=1):
    path = RAW_DIR / f"{session}.mat"
    present = path.is_file()
    actual_md5 = digest_file(path) if present and VERIFY_MD5 else None
    checksum_ok = present and (not VERIFY_MD5 or actual_md5 == EXPECTED_MD5[session])
    raw_rows.append(
        {
            "session": session,
            "split": SPLIT_FOR_SESSION[session],
            "raw_mb": round(path.stat().st_size / 1e6, 1) if present else np.nan,
            "expected_md5": EXPECTED_MD5[session],
            "actual_md5": actual_md5,
            "checksum_ok": checksum_ok,
        }
    )
    print(f"[{index:02d}/37] {session}: {'OK' if checksum_ok else 'CHECK'}")

raw_inventory = pd.DataFrame(raw_rows)
missing_sessions = raw_inventory.loc[~raw_inventory["session"].isin(actual_names), "session"].tolist()
checksum_failures = raw_inventory.loc[~raw_inventory["checksum_ok"], "session"].tolist()
unexpected_sessions = sorted(actual_names - set(EXPECTED_SESSIONS))

assert not missing_sessions, f"Missing raw sessions: {missing_sessions}"
assert not checksum_failures, f"Checksum failures: {checksum_failures}"
assert not unexpected_sessions, f"Unexpected raw sessions: {unexpected_sessions}"
print("Raw collection passed: 37/37 official sessions are valid.")

[01/37] indy_20160407_02: OK
[02/37] indy_20160411_01: OK
[03/37] indy_20160411_02: OK
[04/37] indy_20160418_01: OK
[05/37] indy_20160419_01: OK
[06/37] indy_20160420_01: OK
[07/37] indy_20160426_01: OK
[08/37] indy_20160622_01: OK
[09/37] indy_20160624_03: OK
[10/37] indy_20160627_01: OK
[11/37] indy_20160630_01: OK
[12/37] indy_20160915_01: OK
[13/37] indy_20160916_01: OK
[14/37] indy_20160921_01: OK
[15/37] indy_20160927_04: OK
[16/37] indy_20160927_06: OK
[17/37] indy_20160930_02: OK
[18/37] indy_20160930_05: OK
[19/37] indy_20161005_06: OK
[20/37] indy_20161006_02: OK
[21/37] indy_20161007_02: OK
[22/37] indy_20161011_03: OK
[23/37] indy_20161013_03: OK
[24/37] indy_20161014_04: OK
[25/37] indy_20161017_02: OK
[26/37] indy_20161024_03: OK
[27/37] indy_20161025_04: OK
[28/37] indy_20161026_03: OK
[29/37] indy_20161027_03: OK
[30/37] indy_20161206_02: OK
[31/37] indy_20161207_02: OK
[32/37] indy_20161212_02: OK
[33/37] indy_20161220_02: OK
[34/37] indy_20170123_02: OK
[35/37] indy_2

## 2. Convert one raw session

The converter reads only `t`, `finger_pos`, `chan_names`, and `spikes`. It validates the canonical M1 channel order, aggregates all non-empty unit event vectors per electrode, bins the events, and creates the causal two-axis velocity target.

In [14]:
def decode_matlab_text(dataset: h5py.Dataset) -> str:
    values = np.asarray(dataset).reshape(-1)
    return "".join(chr(int(value)) for value in values if int(value))


def read_channel_names(file: h5py.File) -> np.ndarray:
    return np.asarray(
        [decode_matlab_text(file[ref]) for ref in np.asarray(file["chan_names"]).reshape(-1)]
    )


def is_empty_matlab_cell(dataset: h5py.Dataset) -> bool:
    return bool(dataset.attrs.get("MATLAB_empty", 0))


def validate_artifact(arrays) -> None:
    required = {
        "schema_version",
        "session",
        "source_md5",
        "bin_s",
        "counts",
        "velocity",
        "velocity_lowpass_hz",
        "velocity_filter",
        "velocity_difference",
        "kinematic_sampling",
    }
    assert set(arrays.keys()) == required, f"Unexpected artifact keys: {set(arrays.keys()) ^ required}"
    assert str(np.asarray(arrays["schema_version"]).item()) == SCHEMA_VERSION
    assert np.isclose(float(np.asarray(arrays["bin_s"]).item()), BIN_S)
    assert np.isclose(float(np.asarray(arrays["velocity_lowpass_hz"]).item()), VELOCITY_LOWPASS_HZ)
    assert str(np.asarray(arrays["velocity_filter"]).item()) == "causal_forward_butterworth"
    assert str(np.asarray(arrays["velocity_difference"]).item()) == "backward"
    assert str(np.asarray(arrays["kinematic_sampling"]).item()) == "causal_latest_sample_at_bin_end"

    counts = np.asarray(arrays["counts"])
    velocity = np.asarray(arrays["velocity"])
    assert counts.dtype == np.uint8 and counts.ndim == 2 and counts.shape[0] == 96
    assert velocity.dtype == np.float32 and velocity.shape == (counts.shape[1], 2)
    assert np.isfinite(velocity).all()


def transform_session(path: Path, expected_md5: str) -> tuple[dict[str, np.ndarray], dict]:
    with h5py.File(path, "r") as file:
        timestamps = np.asarray(file["t"]).reshape(-1).astype(np.float64)
        if timestamps.size < 2 or not np.all(np.diff(timestamps) > 0):
            raise ValueError(f"Invalid timestamps: {path.name}")

        finger_position = np.asarray(file["finger_pos"])[:3].T.astype(np.float64)
        channel_names = read_channel_names(file)
        m1_source_indices = np.flatnonzero(np.char.startswith(channel_names, "M1 "))
        m1_names = channel_names[m1_source_indices]
        if not np.array_equal(m1_names, EXPECTED_CHANNEL_NAMES):
            raise ValueError(f"Unexpected M1 channel order: {path.name}")

        n_bins = int(np.floor((timestamps[-1] - timestamps[0]) / BIN_S))
        edges = timestamps[0] + np.arange(n_bins + 1, dtype=np.float64) * BIN_S
        bin_end_times = edges[1:]

        spikes = file["spikes"]
        count_accumulator = np.zeros((96, n_bins), dtype=np.uint16)
        for output_channel, source_channel in enumerate(m1_source_indices):
            for unit in range(spikes.shape[0]):
                cell = file[spikes[unit, source_channel]]
                if is_empty_matlab_cell(cell):
                    continue
                events = np.asarray(cell).reshape(-1)
                if events.size:
                    count_accumulator[output_channel] += np.histogram(events, bins=edges)[0].astype(np.uint16)

    max_count = int(count_accumulator.max(initial=0))
    if max_count > np.iinfo(np.uint8).max:
        raise OverflowError(f"{path.stem}: maximum count {max_count} exceeds uint8")

    sampled_position = causal_sample_hold(timestamps, finger_position, bin_end_times)
    velocity_xyz = causal_velocity(sampled_position, BIN_S, VELOCITY_LOWPASS_HZ)
    velocity_xy = velocity_xyz[:, 1:3].astype(np.float32)

    arrays = {
        "schema_version": np.asarray(SCHEMA_VERSION),
        "session": np.asarray(path.stem),
        "source_md5": np.asarray(expected_md5),
        "bin_s": np.asarray(BIN_S, dtype=np.float32),
        "counts": count_accumulator.astype(np.uint8),
        "velocity": velocity_xy,
        "velocity_lowpass_hz": np.asarray(VELOCITY_LOWPASS_HZ, dtype=np.float32),
        "velocity_filter": np.asarray("causal_forward_butterworth"),
        "velocity_difference": np.asarray("backward"),
        "kinematic_sampling": np.asarray("causal_latest_sample_at_bin_end"),
    }
    validate_artifact(arrays)
    summary = {
        "session": path.stem,
        "bins": n_bins,
        "duration_s": float(n_bins * BIN_S),
        "max_count": max_count,
    }
    return arrays, summary

## 3. Write the split artifacts

Each session is written atomically to its configured split folder. `OVERWRITE=True` replaces artifacts from the earlier metadata-rich schema. The root inventory and manifest record the split, checksums, channel order, target definition, and processing contract.

In [15]:
def atomic_savez(path: Path, arrays: dict[str, np.ndarray]) -> None:
    temporary = path.with_suffix(path.suffix + ".partial")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    with np.load(temporary, allow_pickle=False) as saved:
        validate_artifact(saved)
    temporary.replace(path)


def atomic_json(path: Path, payload: dict) -> None:
    temporary = path.with_suffix(path.suffix + ".partial")
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    temporary.replace(path)


def summarize_artifact(path: Path) -> dict:
    with np.load(path, allow_pickle=False) as artifact:
        validate_artifact(artifact)
        return {
            "session": str(np.asarray(artifact["session"]).item()),
            "split": path.parent.name,
            "artifact": str(path.relative_to(ROOT)),
            "artifact_mb": round(path.stat().st_size / 1e6, 2),
            "bins": int(artifact["counts"].shape[1]),
            "duration_s": float(artifact["counts"].shape[1] * BIN_S),
            "max_count": int(np.asarray(artifact["counts"]).max(initial=0)),
        }


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for split in SPLITS:
    (OUTPUT_DIR / split).mkdir(parents=True, exist_ok=True)

processed_rows = []
batch_started = time.time()

for index, session in enumerate(EXPECTED_SESSIONS, start=1):
    split = SPLIT_FOR_SESSION[session]
    source_path = RAW_DIR / f"{session}.mat"
    artifact_path = OUTPUT_DIR / split / f"{session}.npz"
    if artifact_path.exists() and not OVERWRITE:
        summary = summarize_artifact(artifact_path)
        action = "validated existing"
    else:
        arrays, _ = transform_session(source_path, EXPECTED_MD5[session])
        atomic_savez(artifact_path, arrays)
        summary = summarize_artifact(artifact_path)
        action = "wrote"
    processed_rows.append(summary)
    print(f"[{index:02d}/37] {action}: {split}/{artifact_path.name}")

processed_inventory = pd.DataFrame(processed_rows)
session_inventory = raw_inventory.merge(processed_inventory, on=["session", "split"], how="inner")
session_inventory.to_csv(OUTPUT_DIR / "session_inventory.csv", index=False)

artifact_sha256 = {
    session: digest_file(
        OUTPUT_DIR / SPLIT_FOR_SESSION[session] / f"{session}.npz",
        algorithm="sha256",
    )
    for session in EXPECTED_SESSIONS
}
manifest = {
    "schema_version": SCHEMA_VERSION,
    "source": registry["source"],
    "subject": "indy",
    "session_count": 37,
    "split_strategy": "chronological_session_level_29_4_4",
    "splits": SPLITS,
    "split_counts": {name: len(sessions) for name, sessions in SPLITS.items()},
    "model_input": "96 M1 spike-event counts per 40 ms bin; downstream model selects channels",
    "input_dtype": "uint8",
    "input_channel_order": EXPECTED_CHANNEL_NAMES.tolist(),
    "training_target": "two-dimensional fingertip velocity (-x, -y), cm/s",
    "target_dtype": "float32",
    "bin_s": BIN_S,
    "velocity_lowpass_hz": VELOCITY_LOWPASS_HZ,
    "velocity_filter": "causal_forward_butterworth",
    "velocity_difference": "backward",
    "kinematic_sampling": "causal_latest_sample_at_bin_end",
    "input_smoothing": None,
    "normalization": None,
    "channel_selection": None,
    "raw_md5": EXPECTED_MD5,
    "artifact_sha256": artifact_sha256,
}
atomic_json(OUTPUT_DIR / "manifest.json", manifest)

display(processed_inventory.groupby("split").agg(sessions=("session", "count"), total_bins=("bins", "sum")))
print(f"Batch elapsed: {(time.time() - batch_started) / 60:.1f} minutes")

[01/37] wrote: train/indy_20160407_02.npz
[02/37] wrote: train/indy_20160411_01.npz
[03/37] wrote: train/indy_20160411_02.npz
[04/37] wrote: train/indy_20160418_01.npz
[05/37] wrote: train/indy_20160419_01.npz
[06/37] wrote: train/indy_20160420_01.npz
[07/37] wrote: train/indy_20160426_01.npz
[08/37] wrote: train/indy_20160622_01.npz
[09/37] wrote: train/indy_20160624_03.npz
[10/37] wrote: train/indy_20160627_01.npz
[11/37] wrote: train/indy_20160630_01.npz
[12/37] wrote: train/indy_20160915_01.npz
[13/37] wrote: train/indy_20160916_01.npz
[14/37] wrote: train/indy_20160921_01.npz
[15/37] wrote: train/indy_20160927_04.npz
[16/37] wrote: train/indy_20160927_06.npz
[17/37] wrote: train/indy_20160930_02.npz
[18/37] wrote: train/indy_20160930_05.npz
[19/37] wrote: train/indy_20161005_06.npz
[20/37] wrote: train/indy_20161006_02.npz
[21/37] wrote: train/indy_20161007_02.npz
[22/37] wrote: train/indy_20161011_03.npz
[23/37] wrote: train/indy_20161013_03.npz
[24/37] wrote: train/indy_20161014

,sessions,total_bins
split,,
test,4,68731
train,29,602990
validation,4,57999


Batch elapsed: 0.3 minutes


## 4. Validate the stored dataset

The final check reopens every artifact, enforces the minimal schema, and confirms the exact 29/4/4 folder mapping.

In [16]:
produced_paths = list(OUTPUT_DIR.glob("*/*.npz"))
assert len(produced_paths) == 37, f"Expected 37 artifacts, found {len(produced_paths)}"
assert not list(OUTPUT_DIR.glob("indy_*.npz")), "Artifacts must be inside split folders"

produced_mapping = {path.stem: path.parent.name for path in produced_paths}
assert produced_mapping == SPLIT_FOR_SESSION, "At least one artifact is in the wrong split"

validation_rows = [
    summarize_artifact(OUTPUT_DIR / SPLIT_FOR_SESSION[session] / f"{session}.npz")
    for session in EXPECTED_SESSIONS
]
validation = pd.DataFrame(validation_rows)
assert validation["split"].value_counts().to_dict() == {
    "train": 29,
    "validation": 4,
    "test": 4,
}
assert (OUTPUT_DIR / "session_inventory.csv").is_file()
assert (OUTPUT_DIR / "manifest.json").is_file()

display(validation.groupby("split").agg(sessions=("session", "count"), total_bins=("bins", "sum"), max_count=("max_count", "max")))
print("Validation passed: train=29, validation=4, test=4; all 37 minimal artifacts are valid.")

,sessions,total_bins,max_count
split,,,
test,4,68731,6
train,29,602990,10
validation,4,57999,5


Validation passed: train=29, validation=4, test=4; all 37 minimal artifacts are valid.
